#### insert DB
- 목적
    - 단어의 원형이며, 품사가 지켜진상태에서 원형을 유지하며
    - 부사 -> 형용사 엑셀파일에 맞춰서 변형함.
- DB에 시험 추가시 400번줄에 추가할 시험만 위치 주소 넣으면됨
    - 현재 17~25년 까지 고1,2,3 학, 모, 수가 들어가 있음
    - 추가할 시험만 따로 폴더에 모아서 path 넣으면 됨
- 401번줄
    - 부사 -> 형용사 엑셀파일이며 천부장님 제공한 파일임
    - !!! 만약 수단비 단어가 바뀐다면, 이 파일도 교체되어야함

- ~~gtx970 기준 정상작동하는 라이브러리 버전~~
    - toml 파일 생성하여 dependency library 기록

#### 260414 폴더 구조 수정
- 폴더 분리
- DB 적재 이전과 이후의 코드, 소스등 완전 분리
- PC사양의 변경 (2603월 중순) 으로 전체 라이브러리 수정

### DataBase 내에 table 생성된 상태가 전제되어야 함
- DB접속 정보, 파일 위치정보가 일치해야함
- 빈도를 추출한 단어 엑셀 파일은 "word" 폴더에 위치해야하고, 현재 "영어"라는 column으로 지정되어있음
-
#### 260422 추가사항 기록
- 요청에 대응하다 정리하게된 사항
    - DB내 table은 4개이다.
        - import_log
        - question_item
        - question_token        -- 여기까지 지문, 선지에서 단어, 품사화 기록이며 아래 첫번째 코드는 여기까지만 사용
        - question_token_norm   -- 부사->형용사 규칙에 의한 단어 가공 테이블

In [4]:
from __future__ import annotations

from pathlib import Path
from typing import List, Tuple, Optional, Dict
import re
import csv
import traceback

import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "True"
import mysql.connector
from mysql.connector import MySQLConnection
import spacy


# -----------------------------
# 0) 정규식/전처리 규칙
# -----------------------------

# 문항 구분: 2개 이상 개행(중간 공백 허용)
SPLIT_RE = re.compile(r"(?:\r?\n\s*){2,}")

# POS용: 영어 + 최소 문장부호 유지
KEEP_FOR_POS_RE = re.compile(r"[^A-Za-z\s'\-.,?!:;]+")

# 최종 필터: "단어"만 남김 (2글자 이상, 영문자만)
WORD_RE = re.compile(r"^[A-Za-z]{2,}$")

# 파일명 파싱: (H|M|S)(1|2|3)_(YY)_(MM).txt
FNAME_RE = re.compile(r"^(?P<type>[HMS])(?P<grade>[123])_(?P<yy>\d{2})_(?P<mm>\d{2})$", re.IGNORECASE)


# -----------------------------
# 1) 유틸
# -----------------------------

def safe_var_name(stem: str) -> str:
    name = re.sub(r"[^0-9a-zA-Z_]", "_", stem)
    if not name or name[0].isdigit():
        name = "_" + name
    return name


def normalize_for_pos(text: str) -> str:
    text = KEEP_FOR_POS_RE.sub(" ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def parse_questions_blocks(text: str) -> List[str]:
    text = text.strip()
    if not text:
        return []
    return [b.strip() for b in SPLIT_RE.split(text) if b.strip()]


def q_key_and_range(q: int) -> Tuple[str, int, int, int]:
    """
    18부터 시작해서 41→41-42, 43→43-45 묶기
    """
    if q == 41:
        return "41-42", 41, 42, 2
    if q == 43:
        return "43-45", 43, 45, 3
    return str(q), q, q, 1


def parse_exam_from_filename(stem: str) -> Tuple[str, int, int, int]:
    """
    예) H1_17_03 -> (학평, 1, 2017, 3)
        M3_22_06 -> (모평, 3, 2022, 6)
        S3_20_11 -> (수능, 3, 2020, 11)
    """
    m = FNAME_RE.match(stem)
    if not m:
        raise ValueError(f"파일명 패턴 불일치: {stem} (예: H1_17_03.txt)")

    t = m.group("type").upper()
    grade = int(m.group("grade"))
    yy = int(m.group("yy"))
    mm = int(m.group("mm"))

    exam_type = {"H": "학평", "M": "모평", "S": "수능"}[t]
    year = 2000 + yy  # 00~99는 2000년대로 가정 (필요하면 1990대 규칙 추가 가능)
    month = mm

    return exam_type, grade, year, month


def load_adv_to_adj_map(csv_path: str | Path) -> Dict[str, str]:
    """
    CSV에서
      - key: '수단비(부)' (부사)
      - value: '수단비' (형용사)
    맵 생성 (소문자 기준)
    """
    csv_path = Path(csv_path)
    adv_map: Dict[str, str] = {}

    with csv_path.open("r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            adv = (row.get("수단비(부)") or "").strip()
            adj = (row.get("수단비") or "").strip()
            if not adv or not adj:
                continue
            adv_map[adv.lower()] = adj.lower()

    return adv_map


# -----------------------------
# 2) MySQL
# -----------------------------

def connect_mysql(host: str, port: int, user: str, password: str, database: str) -> MySQLConnection:
    return mysql.connector.connect(
        host=host,
        port=port,
        user=user,
        password=password,
        database=database,
        autocommit=False,
        charset="utf8mb4",
        collation="utf8mb4_unicode_ci",
    )


def upsert_question_item(
    cnx: MySQLConnection,
    exam_type: str,
    grade: int,
    exam_year: int,
    exam_month: int,
    file_name: str,
    q_key: str,
    block_index: int,
    content_raw: str,
    content_for_pos: str,
    content_en_clean: str,
    token_count: int,
) -> int:
    sql = """
    INSERT INTO question_item
      (exam_type, grade, exam_year, exam_month,
       file_name, q_key, block_index,
       content_raw, content_for_pos, content_en_clean, token_count)
    VALUES
      (%s, %s, %s, %s,
       %s, %s, %s,
       %s, %s, %s, %s)
    ON DUPLICATE KEY UPDATE
      exam_type = VALUES(exam_type),
      grade = VALUES(grade),
      exam_year = VALUES(exam_year),
      exam_month = VALUES(exam_month),
      block_index = VALUES(block_index),
      content_raw = VALUES(content_raw),
      content_for_pos = VALUES(content_for_pos),
      content_en_clean = VALUES(content_en_clean),
      token_count = VALUES(token_count),
      updated_at = CURRENT_TIMESTAMP
    """
    cur = cnx.cursor()
    cur.execute(
        sql,
        (
            exam_type, grade, exam_year, exam_month,
            file_name, q_key, block_index,
            content_raw, content_for_pos, content_en_clean, token_count
        ),
    )

    cur.execute(
        """
        SELECT id FROM question_item WHERE file_name=%s AND q_key=%s
        """,
        (file_name, q_key),
    )
    row = cur.fetchone()
    if not row:
        raise RuntimeError("UPSERT 후 id 조회 실패")
    return int(row[0])


def replace_tokens(
    cnx: MySQLConnection,
    question_id: int,
    tokens: List[Tuple[int, str, Optional[str], Optional[str], Optional[str], int, Optional[int], Optional[int]]],
) -> None:
    cur = cnx.cursor()
    cur.execute("DELETE FROM question_token WHERE question_id=%s", (question_id,))

    if not tokens:
        return

    ins = """
    INSERT INTO question_token
      (question_id, token_index, text, lemma, pos, tag, dep, is_stop, start_char, end_char)
    VALUES
      (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    """
    payload = [
        (question_id, ti, tx, lm, ps, tg, None, st, sc, ec)
        for (ti, tx, lm, ps, tg, st, sc, ec) in tokens
    ]
    cur.executemany(ins, payload)


def write_import_log(
    cnx: MySQLConnection,
    file_name: str,
    exam_type: str,
    grade: int,
    exam_year: int,
    exam_month: int,
    question_count: int,
    warn_not_25: int,
    note: Optional[str] = None,
) -> None:
    cur = cnx.cursor()
    cur.execute(
        """
        INSERT INTO import_log (file_name, exam_type, grade, exam_year, exam_month, question_count, warn_not_25, note)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
        """,
        (file_name, exam_type, grade, exam_year, exam_month, question_count, warn_not_25, note),
    )


# -----------------------------
# 3) 메인 임포트
# -----------------------------

def import_folder(
    folder_path: str | Path,
    cnx: MySQLConnection,
    adv_csv_path: str | Path,
    start_q: int = 18,
    file_pattern: str = "*.txt",
    spacy_model: str = "en_core_web_lg",
) -> None:
    folder_path = Path(folder_path)
    folder_name = folder_path.name

    # 경로/파일 확인용
    print("[DEBUG] folder_path =", folder_path.resolve())
    print("[DEBUG] exists =", folder_path.exists(), "\t\tis_dir =", folder_path.is_dir())

    txt_files = sorted(folder_path.glob(file_pattern))
    print("[DEBUG] pattern =", file_pattern, "found =", len(txt_files))

    adv_map = load_adv_to_adj_map(adv_csv_path)
    print("[DEBUG] adv_map size =", len(adv_map))

    # POS 태깅
    nlp = spacy.load(spacy_model, disable=["ner"])

    for fp in txt_files:
        file_name = fp.name
        stem = fp.stem  # H1_17_03 같은 부분
        var_name = safe_var_name(stem)

        # 3-1) 파일명 파싱 (선행 작업)
        try:
            exam_type, grade, exam_year, exam_month = parse_exam_from_filename(stem)
        except Exception as e:
            print(f"[ERR] 파일명 파싱 실패: {file_name} -> {e}")
            continue

        raw_text = fp.read_text(encoding="utf-8", errors="replace")
        blocks = parse_questions_blocks(raw_text)

        # 1) “25개가 아니면 경고” (요구대로)
        warn = 1 if len(blocks) != 25 else 0
        if warn:
            print(f"[경고] {folder_name}/{file_name}: 블록 {len(blocks)}개 (25개가 아님)")

        q = start_q
        block_index = 0

        try:
            for block in blocks:
                block_index += 1

                q_key, q_start, q_end, step = q_key_and_range(q)
                q += step

                content_raw = block
                content_for_pos = normalize_for_pos(block)

                # POS용 텍스트가 비면 스킵(영어가 없는 블록)
                if not content_for_pos:
                    continue

                doc = nlp(content_for_pos)

                tokens_out = []
                final_words = []

                ti = 0
                for t in doc:
                    if t.is_space:
                        continue

                    text = t.text
                    lemma = t.lemma_ if t.lemma_ else t.text
                    pos = t.pos_ if t.pos_ else None
                    tag = t.tag_ if t.tag_ else None

                    # 2) 부사→형용사 변환 (조건: ADV + CSV '수단비(부)'에 존재)
                    if pos == "ADV":
                        k1 = (lemma or "").lower()
                        k2 = (text or "").lower()
                        conv = adv_map.get(k1) or adv_map.get(k2)
                        if conv:
                            text = conv
                            lemma = conv
                            pos = "ADJ"

                    # 3) 최종 필터: 한 글자 / . , 등 단어 아닌 것 제거
                    #    => 영문자만 2글자 이상만 통과
                    if not WORD_RE.match(text):
                        continue

                    final_words.append(text)
                    tokens_out.append(
                        (
                            ti,
                            text,
                            (lemma.lower() if lemma else None),
                            pos,
                            tag,
                            (1 if t.is_stop else 0),
                            int(t.idx),
                            int(t.idx + len(t.text)),
                        )
                    )
                    ti += 1

                token_count = len(tokens_out)

                # ✅ DB에 넣는 content_en_clean은 "변환+필터 완료된 최종 토큰" 기반
                content_en_clean = " ".join(final_words)

                qid = upsert_question_item(
                    cnx=cnx,
                    exam_type=exam_type,
                    grade=grade,
                    exam_year=exam_year,
                    exam_month=exam_month,
                    file_name=file_name,
                    q_key=q_key,
                    block_index=block_index,
                    content_raw=content_raw,
                    content_for_pos=content_for_pos,
                    content_en_clean=content_en_clean,
                    token_count=token_count,
                )

                replace_tokens(cnx, qid, tokens_out)

            write_import_log(
                cnx=cnx,
                file_name=file_name,
                exam_type=exam_type,
                grade=grade,
                exam_year=exam_year,
                exam_month=exam_month,
                question_count=len(blocks),
                warn_not_25=warn,   # 컬럼명이 warn_over_25지만 현재는 “25개가 아님” 용도로 사용
                note=f"{exam_type}, 고{grade}, {exam_year}-{exam_month:02d}",
            )

            cnx.commit()
            print(f"[OK] {folder_name}/{file_name} 완료 (블록 {len(blocks)}개, {exam_type}/고{grade}/{exam_year}-{exam_month:02d})")

        except Exception as e:
            cnx.rollback()
            print(f"[ERR] {folder_name}/{file_name} 처리 실패: {e}")
            traceback.print_exc()
            continue

    print("\n완료!!!")


if __name__ == "__main__":
    MYSQL = {
        # "host": "127.0.0.1",
        "host": "172.30.1.23",
        "port": 3306,
        "user": "dbadmin",
        "password": "12341234",
        "database": "sudanbi_DB",
    }

    # ✅ 실행 위치에 흔들리지 않게 "이 파일 기준"으로 잡는 걸 추천
    try:
        BASE = Path(__file__).resolve().parent
    except NameError:
        BASE = Path.cwd()   # Jupyter / IPython

    # 추가할 시험
    TARGET_FOLDER = Path("./data/ms")

    # 부사 -> 형용사 DB (천부장님 제공)
    ADV_CSV = Path("./data/generated_adverb_to_adjective_수정_0805.csv")

    cnx = connect_mysql(**MYSQL)
    try:
        import_folder(
            folder_path=TARGET_FOLDER,
            cnx=cnx,
            adv_csv_path=ADV_CSV,
            start_q=18,
            file_pattern="*.txt",
            spacy_model="en_core_web_lg",
        )
    finally:
        cnx.close()


[DEBUG] folder_path = D:\project\knsonline\sudanbi\data\MS
[DEBUG] exists = True 		is_dir = True
[DEBUG] pattern = *.txt found = 27
[DEBUG] adv_map size = 1
[OK] ms/M3_17_06.txt 완료 (블록 25개, 모평/고3/2017-06)
[OK] ms/M3_17_09.txt 완료 (블록 25개, 모평/고3/2017-09)
[OK] ms/M3_18_06.txt 완료 (블록 25개, 모평/고3/2018-06)
[OK] ms/M3_18_09.txt 완료 (블록 25개, 모평/고3/2018-09)
[OK] ms/M3_19_06.txt 완료 (블록 25개, 모평/고3/2019-06)
[OK] ms/M3_19_09.txt 완료 (블록 25개, 모평/고3/2019-09)
[OK] ms/M3_20_06.txt 완료 (블록 25개, 모평/고3/2020-06)
[OK] ms/M3_20_09.txt 완료 (블록 25개, 모평/고3/2020-09)
[OK] ms/M3_21_06.txt 완료 (블록 25개, 모평/고3/2021-06)
[OK] ms/M3_21_09.txt 완료 (블록 25개, 모평/고3/2021-09)
[OK] ms/M3_22_06.txt 완료 (블록 25개, 모평/고3/2022-06)
[OK] ms/M3_22_09.txt 완료 (블록 25개, 모평/고3/2022-09)
[OK] ms/M3_23_06.txt 완료 (블록 25개, 모평/고3/2023-06)
[OK] ms/M3_23_09.txt 완료 (블록 25개, 모평/고3/2023-09)
[OK] ms/M3_24_06.txt 완료 (블록 25개, 모평/고3/2024-06)
[OK] ms/M3_24_09.txt 완료 (블록 25개, 모평/고3/2024-09)
[OK] ms/M3_25_06.txt 완료 (블록 25개, 모평/고3/2025-06)
[OK] ms/M3_25_09.txt 완료 (블록

#### aggregation
- DB에서 엑셀로 꺼내오기
- 148번줄의 WORDBOOK_XLSX 변수에 수단비에 들어가는 단어가 있는 엑셀 파일 path 지정하면 됨

In [10]:
from __future__ import annotations

from pathlib import Path
import pandas as pd
import mysql.connector

TYPE_SHORT = {"학평": "학", "모평": "모", "수능": "수"}

def normalize_word(w: str) -> str:
    return (w or "").strip().lower()

def load_wordbook_words(xlsx_path: str | Path, word_col: str = "word") -> list[str]:
    df = pd.read_excel(xlsx_path)#, sheet_name='중3')
    if word_col not in df.columns:
        raise ValueError(f"엑셀에 '{word_col}' 컬럼이 없습니다. 실제 컬럼: {list(df.columns)}")

    words = [normalize_word(x) for x in df[word_col].tolist()]
    words = [w for w in words if w]

    # 중복 제거(순서 유지)
    seen = set()
    uniq = []
    for w in words:
        if w not in seen:
            uniq.append(w)
            seen.add(w)
    return uniq

def fetch_counts(cnx, exam_types: list[str], grade: int, words: list[str]) -> pd.DataFrame:
    """
    exam_types(예: ['수능','모평']) + grade 조건에서
    단어(word)별 (year,month,exam_type) 토큰 출현 횟수 집계
    """
    if not words:
        return pd.DataFrame(columns=["word", "exam_year", "exam_month", "exam_type", "cnt"])

    type_placeholders = ",".join(["%s"] * len(exam_types))

    sql = f"""
    SELECT
      w.word AS word,
      qi.exam_year AS exam_year,
      qi.exam_month AS exam_month,
      qi.exam_type AS exam_type,
      COUNT(*) AS cnt
    FROM question_token qt
    JOIN question_item qi
      ON qi.id = qt.question_id
    JOIN (
      SELECT %s AS word
      {" UNION ALL SELECT %s" * (len(words) - 1)}
    ) w
      ON (qt.lemma = w.word OR qt.text = w.word)
    WHERE qi.exam_type IN ({type_placeholders})
      AND qi.grade = %s
    GROUP BY w.word, qi.exam_year, qi.exam_month, qi.exam_type
    """

    params = []
    params.extend(words)          # w.word 파생테이블용
    params.extend(exam_types)     # IN (...)용
    params.append(grade)          # grade

    df = pd.read_sql(sql, cnx, params=params)
    if df.empty:
        return df

    df["exam_year"] = df["exam_year"].astype(int)
    df["exam_month"] = df["exam_month"].astype(int)
    df["cnt"] = df["cnt"].astype(int)
    return df

def pivot_to_excel(df_counts: pd.DataFrame, words: list[str], grade: int, out_path: str | Path) -> None:
    """
    출력 컬럼: {year}_{grade}_{typeShort}_{month}
    날짜 역순(year desc, month desc)
    수능/모평 같이 포함 가능(컬럼에 exam_type이 들어가기 때문)
    """
    out_path = Path(out_path)

    if df_counts.empty:
        pd.DataFrame({"word": words}).to_excel(out_path, index=False)
        return

    pv = df_counts.pivot_table(
        index="word",
        columns=["exam_year", "exam_month", "exam_type"],
        values="cnt",
        aggfunc="sum",
        fill_value=0,
    )

    # ✅ 날짜 역순 정렬 (year desc, month desc)
    cols_sorted = sorted(list(pv.columns), key=lambda x: (x[0], x[1]), reverse=True)
    pv = pv[cols_sorted]

    # 컬럼명 포맷: 2025_3_수_11 (month는 2자리)
    pv.columns = [
        f"{y}_{grade}_{TYPE_SHORT.get(t, t)}_{m:02d}"
        for (y, m, t) in pv.columns
    ]

    pv = pv.reset_index()

    # wordbook 기준으로 0 채우기
    all_words_df = pd.DataFrame({"word": words})
    out_df = all_words_df.merge(pv, on="word", how="left").fillna(0)

    for c in out_df.columns[1:]:
        out_df[c] = out_df[c].astype(int)

    out_df.to_excel(out_path, index=False)

def run(
    wordbook_xlsx: str | Path,
    exam_types: list[str],
    grade: int,
    mysql_cfg: dict,
    out_xlsx: str | Path,
    word_col: str = "word",
) -> None:
    words = load_wordbook_words(wordbook_xlsx, word_col=word_col)

    cnx = mysql.connector.connect(**mysql_cfg)
    try:
        df_counts = fetch_counts(cnx, exam_types=exam_types, grade=grade, words=words)
    finally:
        cnx.close()

    pivot_to_excel(df_counts, words=words, grade=grade, out_path=out_xlsx)

if __name__ == "__main__":
    MYSQL = {
        # "host": "127.0.0.1",
        "host": "172.30.1.23",
        "port": 3306,
        "user": "dbadmin",
        "password": "12341234",
        "database": "sudanbi_DB",
    }

    WORDBOOK_XLSX = r"C:\automation\Request_RND\SuDanBi\250310 수능 어휘집\data\필수편.xlsx"
    EXAM_TYPES = ["학평", "모평", "수능"]  # ✅ 리스트
    GRADE = 3

    OUT_XLSX = f"./result/필수편{'_'.join(EXAM_TYPES)}_고{GRADE}.xlsx"

    run(WORDBOOK_XLSX, EXAM_TYPES, GRADE, MYSQL, OUT_XLSX)
    print("saved:", OUT_XLSX)


C:\Users\OBDTS\AppData\Local\Temp\ipykernel_20648\138744497.py:64: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, cnx, params=params)


saved: C:\automation\Request_RND\SuDanBi\250310 수능 어휘집\251217 작업 결과물\필수편학평_모평_수능_고3.xlsx


150번 줄 GRADE 값이 DB에 저장된 학년을 의미한다.

In [ ]:
# QUESTION_TOKEN table에서 text 열 -> 품사 무시 원형화 하여 새로운 테이블 question_token_norm 에 저장 ㄱㄱ

In [3]:
import re
import mysql.connector

WORD_RE = re.compile(r"^[A-Za-z]{2,}$")

def norm_lemma(text: str, lemma: str | None, pos: str | None) -> tuple[str, str]:
    t = (text or "").lower().strip()
    l = (lemma or "").lower().strip() or t

    if (pos == "ADJ") and (l == t) and t.endswith("ed"):
        if t.endswith("ied") and len(t) > 3:
            return (t[:-3] + "y", "AdjEdToVerb_ied")
        if len(t) > 3:
            return (t[:-2], "AdjEdToVerb_ed")

    return (l, "SpaCyLemmaOrFallback")


def main():
    MYSQL = {
        "host": "172.30.1.23",
        "port": 3306,
        "user": "dbadmin",
        "password": "12341234",
        "database": "sudanbi_DB",
    }

    # ✅ 읽기/쓰기 커넥션 분리
    cnx_r = mysql.connector.connect(**MYSQL)
    cnx_w = mysql.connector.connect(**MYSQL)
    cnx_w.autocommit = False

    sel = """
    SELECT id, question_id, token_index, text, lemma, pos
    FROM question_token
    ORDER BY id
    """
    ins = """
    INSERT INTO question_token_norm
      (token_id, question_id, token_index, text, lemma_norm, rule)
    VALUES
      (%s, %s, %s, %s, %s, %s)
    ON DUPLICATE KEY UPDATE
      question_id = VALUES(question_id),
      token_index = VALUES(token_index),
      text = VALUES(text),
      lemma_norm = VALUES(lemma_norm),
      rule = VALUES(rule)
    """

    cur_r = cnx_r.cursor()     # 읽기 커서
    cur_w = cnx_w.cursor()     # 쓰기 커서

    cur_r.execute(sel)

    batch = []
    n = 0

    for token_id, question_id, token_index, text, lemma, pos in cur_r:
        if not WORD_RE.match((text or "")):
            continue

        lemma_norm, rule = norm_lemma(text, lemma, pos)
        batch.append((token_id, question_id, token_index, text, lemma_norm, rule))
        n += 1

        if len(batch) >= 5000:
            cur_w.executemany(ins, batch)
            cnx_w.commit()
            batch.clear()
            print("[OK] wrote", n)

    if batch:
        cur_w.executemany(ins, batch)
        cnx_w.commit()
        print("[OK] wrote", n)

    cur_r.close()
    cur_w.close()
    cnx_r.close()
    cnx_w.close()


if __name__ == "__main__":
    main()


[OK] wrote 5000
[OK] wrote 10000
[OK] wrote 15000
[OK] wrote 20000
[OK] wrote 25000
[OK] wrote 30000
[OK] wrote 35000
[OK] wrote 40000
[OK] wrote 45000
[OK] wrote 50000
[OK] wrote 55000
[OK] wrote 60000
[OK] wrote 65000
[OK] wrote 70000
[OK] wrote 75000
[OK] wrote 80000
[OK] wrote 85000
[OK] wrote 90000
[OK] wrote 95000
[OK] wrote 100000
[OK] wrote 105000
[OK] wrote 110000
[OK] wrote 115000
[OK] wrote 120000
[OK] wrote 125000
[OK] wrote 130000
[OK] wrote 135000
[OK] wrote 140000
[OK] wrote 145000
[OK] wrote 150000
[OK] wrote 155000
[OK] wrote 160000
[OK] wrote 165000
[OK] wrote 170000
[OK] wrote 175000
[OK] wrote 180000
[OK] wrote 185000
[OK] wrote 190000
[OK] wrote 195000
[OK] wrote 200000
[OK] wrote 205000
[OK] wrote 210000
[OK] wrote 215000
[OK] wrote 220000
[OK] wrote 225000
[OK] wrote 230000
[OK] wrote 235000
[OK] wrote 240000
[OK] wrote 245000
[OK] wrote 250000
[OK] wrote 255000
[OK] wrote 260000
[OK] wrote 265000
[OK] wrote 270000
[OK] wrote 275000
[OK] wrote 280000
[OK] wrote 2

In [ ]:
## 원형화된 테이블에서 수단비 뽑아내는 코드

In [9]:
from __future__ import annotations

from pathlib import Path
import pandas as pd
import mysql.connector

TYPE_SHORT = {"학평": "학", "모평": "모", "수능": "수"}


def normalize_word(w: str) -> str:
    return (w or "").strip().lower()


def load_wordbook_words(xlsx_path: str | Path, word_col: str = "word") -> list[str]:
    df = pd.read_excel(xlsx_path)  #, sheet_name='중3')
    if word_col not in df.columns:
        raise ValueError(f"엑셀에 '{word_col}' 컬럼이 없습니다. 실제 컬럼: {list(df.columns)}")

    words = [normalize_word(x) for x in df[word_col].tolist()]
    words = [w for w in words if w]

    # 중복 제거(순서 유지)
    seen = set()
    uniq = []
    for w in words:
        if w not in seen:
            uniq.append(w)
            seen.add(w)
    return uniq


def fetch_counts(cnx, exam_types: list[str], grade: int, words: list[str]) -> pd.DataFrame:
    if not words:
        return pd.DataFrame(columns=["word", "exam_year", "exam_month", "exam_type", "cnt"])

    type_placeholders = ",".join(["%s"] * len(exam_types))

    sql = f"""
    SELECT
      w.word AS word,
      qi.exam_year AS exam_year,
      qi.exam_month AS exam_month,
      qi.exam_type AS exam_type,
      COUNT(*) AS cnt
    FROM question_token_norm qtn
    JOIN question_item qi
      ON qi.id = qtn.question_id
    JOIN (
      SELECT %s AS word
      {" UNION ALL SELECT %s" * (len(words) - 1)}
    ) w
      ON qtn.lemma_norm = w.word
    WHERE qi.exam_type IN ({type_placeholders})
      AND qi.grade = %s
    GROUP BY w.word, qi.exam_year, qi.exam_month, qi.exam_type
    """

    params = []
    params.extend(words)  # w.word 파생테이블용
    params.extend(exam_types)  # IN (...)용
    params.append(grade)  # grade

    df = pd.read_sql(sql, cnx, params=params)
    if df.empty:
        return df

    df["exam_year"] = df["exam_year"].astype(int)
    df["exam_month"] = df["exam_month"].astype(int)
    df["cnt"] = df["cnt"].astype(int)
    return df


def pivot_to_excel(df_counts: pd.DataFrame, words: list[str], grade: int, out_path: str | Path) -> None:
    """
    출력 컬럼: {year}_{grade}_{typeShort}_{month}
    날짜 역순(year desc, month desc)
    수능/모평 같이 포함 가능(컬럼에 exam_type이 들어가기 때문)
    """
    out_path = Path(out_path)

    if df_counts.empty:
        pd.DataFrame({"word": words}).to_excel(out_path, index=False)
        return

    pv = df_counts.pivot_table(
        index="word",
        columns=["exam_year", "exam_month", "exam_type"],
        values="cnt",
        aggfunc="sum",
        fill_value=0,
    )

    # ✅ 날짜 역순 정렬 (year desc, month desc)
    cols_sorted = sorted(list(pv.columns), key=lambda x: (x[0], x[1]), reverse=True)
    pv = pv[cols_sorted]

    # 컬럼명 포맷: 2025_3_수_11 (month는 2자리)
    pv.columns = [
        f"{y}_{grade}_{TYPE_SHORT.get(t, t)}_{m:02d}"
        for (y, m, t) in pv.columns
    ]

    pv = pv.reset_index()

    # wordbook 기준으로 0 채우기
    all_words_df = pd.DataFrame({"word": words})
    out_df = all_words_df.merge(pv, on="word", how="left").fillna(0)

    for c in out_df.columns[1:]:
        out_df[c] = out_df[c].astype(int)

    out_df.to_excel(out_path, index=False)


def run(
        wordbook_xlsx: str | Path,
        exam_types: list[str],
        grade: int,
        mysql_cfg: dict,
        out_xlsx: str | Path,
        word_col: str = "영어",
) -> None:
    words = load_wordbook_words(wordbook_xlsx, word_col=word_col)

    cnx = mysql.connector.connect(**mysql_cfg)
    try:
        df_counts = fetch_counts(cnx, exam_types=exam_types, grade=grade, words=words)
    finally:
        cnx.close()

    pivot_to_excel(df_counts, words=words, grade=grade, out_path=out_xlsx)


if __name__ == "__main__":
    MYSQL = {
        "host": "172.30.1.23",
        "port": 3306,
        "user": "dbadmin",
        "password": "12341234",
        "database": "sudanbi_DB",
    }


    WORDBOOK_XLSX = "./word/고23new.xlsx"
    EXAM_TYPES = ["학평", "모평", "수능"]  # ✅ 리스트
    GRADE = 1

    OUT_XLSX = f"./result/{'_'.join(EXAM_TYPES)}_고{GRADE}.xlsx"

    run(WORDBOOK_XLSX, EXAM_TYPES, GRADE, MYSQL, OUT_XLSX)
    print("saved:", OUT_XLSX)


C:\Users\OBDTS\AppData\Local\Temp\ipykernel_23416\4252780320.py:63: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, cnx, params=params)


saved: ./result/학평_모평_수능_고1.xlsx
